# Current Modal Run Monitor (Apr 10)

This notebook is for live monitoring of the current previous-V2 duplicate runs launched by `~/tom-run` or `~/tom-run-detached`.

It is designed to work both:
- while the run is still in progress and only `run_status.json` / `progress.json` / `stdout.log` exist, and
- after final `run_summary.json` files have been written.

By default it watches the volume for `check2`, but you can override that with environment variables:
- `TOM_RUN_STAMP=...`
- `TOM_VOLUME_NAME=...`
- `TOM_TARGET_TOTAL_EPISODES=...`


In [ ]:
import csv
import io
import json
import os
import subprocess
from pathlib import Path, PurePosixPath

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    import pandas as pd
except ImportError:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        if all((base / name).exists() for name in ('train.py', 'env.py', 'eval.py')):
            return base
    fallback = Path('/Users/stephenbeale/Projects/ToM_AI_Research_Team')
    if all((fallback / name).exists() for name in ('train.py', 'env.py', 'eval.py')):
        return fallback
    raise FileNotFoundError('Could not locate repo root from current notebook context.')


REPO_ROOT = find_repo_root()
RUN_ID_OR_VOLUME = os.environ.get('TOM_VOLUME_NAME') or os.environ.get('TOM_RUN_STAMP', 'check2')
VOLUME_NAME = (
    RUN_ID_OR_VOLUME
    if RUN_ID_OR_VOLUME.startswith('tom-previous-v2-duplicate-')
    else f'tom-previous-v2-duplicate-{RUN_ID_OR_VOLUME}'
)
RUN_ROOT = PurePosixPath(os.environ.get('TOM_OUTPUT_FAMILY', 'auxhead-lite'))
REMOTE_OUTPUT_PREFIX = PurePosixPath('/root/outputs')
DEFAULT_TARGET_TOTAL_EPISODES = int(os.environ.get('TOM_TARGET_TOTAL_EPISODES', '140000'))
INITIAL_800_ANALYSIS_ROOT = REPO_ROOT / 'modal' / 'initial-800-run-analysis'
RESULTS_130_ROOT = REPO_ROOT / 'modal' / 'tom-130k-modal-results'
RESULTS_140_OLD_ROOT = REPO_ROOT / 'modal' / 'tom-140k-modal-results'
RESULTS_140_V2_ROOT = REPO_ROOT / 'modal' / 'tom-140k-modal-results-v2'
SEEDS = [7, 11]


def modal_cli(*args, allow_error=False):
    proc = subprocess.run(['modal', *[str(arg) for arg in args]], capture_output=True, text=True)
    if proc.returncode != 0 and not allow_error:
        raise RuntimeError((proc.stderr or proc.stdout).strip())
    return proc.stdout, proc


def volume_ls_json(path: str | PurePosixPath):
    stdout, proc = modal_cli('volume', 'ls', '--json', VOLUME_NAME, str(path), allow_error=True)
    if proc.returncode != 0:
        return []
    return json.loads(stdout)


def read_text(path: str | PurePosixPath) -> str:
    stdout, proc = modal_cli('volume', 'get', VOLUME_NAME, str(path), '-', allow_error=True)
    if proc.returncode != 0:
        raise FileNotFoundError((proc.stderr or proc.stdout).strip())
    return stdout.replace('✓ Finished downloading files to local!\n', '').rstrip()


def read_text_if_exists(path: str | PurePosixPath) -> str | None:
    try:
        return read_text(path)
    except Exception:
        return None


def read_json_if_exists(path: str | PurePosixPath):
    text = read_text_if_exists(path)
    return json.loads(text) if text else None


def tail_text(path: str | PurePosixPath, lines: int = 40) -> str:
    text = read_text_if_exists(path)
    if not text:
        return '<missing>'
    return '\n'.join(text.splitlines()[-lines:])


def remote_to_volume_path(remote_path: str) -> PurePosixPath:
    remote = PurePosixPath(remote_path)
    if remote.is_absolute():
        return remote.relative_to(REMOTE_OUTPUT_PREFIX)
    return remote


def as_table(rows, sort_by=None, ascending=False):
    if pd is None:
        return rows
    df = pd.DataFrame(rows)
    if sort_by is not None and len(df):
        df = df.sort_values(sort_by, ascending=ascending)
    return df


print(f'repo_root={REPO_ROOT}')
print(f'volume={VOLUME_NAME}')
print(f'run_root={RUN_ROOT}')


In [ ]:
def seed_target_totals(seed: int):
    rows = volume_ls_json(RUN_ROOT / f'seed{seed}')
    targets = []
    for row in rows:
        name = PurePosixPath(row['Filename']).name
        if name.startswith('target-'):
            try:
                targets.append(int(name.removeprefix('target-')))
            except ValueError:
                pass
    return sorted(set(targets))


def run_dir(seed: int, target_total_episodes: int | None = None) -> PurePosixPath:
    target_total_episodes = target_total_episodes or latest_target_total(seed=seed)
    return RUN_ROOT / f'seed{seed}' / f'target-{target_total_episodes}'


def latest_curve_path(seed: int, target_total_episodes: int | None = None) -> PurePosixPath:
    target_total_episodes = target_total_episodes or latest_target_total(seed=seed)
    return run_dir(seed, target_total_episodes) / 'curves' / f'latest-curve-tom-seed{seed}.csv'


def load_summaries():
    rows = []
    for seed in SEEDS:
        for target_total_episodes in seed_target_totals(seed):
            path = run_dir(seed, target_total_episodes) / 'run_summary.json'
            payload = read_json_if_exists(path)
            if not payload:
                continue
            eval_metrics = payload.get('eval_metrics') or {}
            rows.append(
                {
                    'seed': payload.get('seed'),
                    'target_total_episodes': payload.get('target_total_episodes'),
                    'additional_train_episodes': payload.get('additional_train_episodes'),
                    'initial_checkpoint_train_episodes': payload.get('initial_checkpoint_train_episodes'),
                    'returncode': payload.get('returncode'),
                    'path': str(path),
                    'output_dir': payload.get('output_dir'),
                    'ToMCoordScore': eval_metrics.get('ToMCoordScore'),
                    'SuccessRate': eval_metrics.get('SuccessRate'),
                    'CollisionRate': eval_metrics.get('CollisionRate'),
                    'DeadlockRate': eval_metrics.get('DeadlockRate'),
                    'IntentionPredictionF1': eval_metrics.get('IntentionPredictionF1'),
                    'StrategySwitchAccuracy': eval_metrics.get('StrategySwitchAccuracy'),
                    'AmbiguityEfficiency': eval_metrics.get('AmbiguityEfficiency'),
                    'AverageDelay': eval_metrics.get('AverageDelay'),
                }
            )
    return sorted(rows, key=lambda row: (row['target_total_episodes'], row['seed']))


def load_status_rows():
    rows = []
    for seed in SEEDS:
        for target_total_episodes in seed_target_totals(seed):
            base = run_dir(seed, target_total_episodes)
            status = read_json_if_exists(base / 'run_status.json') or {}
            progress = read_json_if_exists(base / 'progress.json') or {}
            summary = read_json_if_exists(base / 'run_summary.json') or {}
            eval_metrics = summary.get('eval_metrics') or {}
            rows.append(
                {
                    'seed': seed,
                    'target_total_episodes': target_total_episodes,
                    'state': status.get('state'),
                    'completed_total_episodes': progress.get('completed_total_episodes', status.get('completed_total_episodes')),
                    'remaining_total_episodes': progress.get('remaining_total_episodes', status.get('remaining_episodes')),
                    'completed_additional_episodes': progress.get('completed_additional_episodes', status.get('completed_additional_episodes')),
                    'is_final': progress.get('is_final'),
                    'returncode': summary.get('returncode'),
                    'path': str(base),
                    'ToMCoordScore': eval_metrics.get('ToMCoordScore'),
                }
            )
    return sorted(rows, key=lambda row: (row['target_total_episodes'], row['seed']))


def latest_target_total(seed: int | None = None) -> int:
    targets = []
    for candidate_seed in SEEDS if seed is None else [seed]:
        targets.extend(seed_target_totals(candidate_seed))
    if not targets:
        raise ValueError('No target directories found in the Modal volume yet.')
    return max(targets)


discovered_targets = {seed: seed_target_totals(seed) for seed in SEEDS}
summaries = load_summaries()
status_rows = load_status_rows()
print(f'discovered_targets={discovered_targets}')
print(f'completed_summaries={len(summaries)}')
print(f'status_rows={len(status_rows)}')
as_table(status_rows, sort_by=['target_total_episodes', 'seed'], ascending=True)


In [ ]:
SEED = 7
TARGET_TOTAL_EPISODES = latest_target_total(seed=SEED)

selected_run_dir = run_dir(SEED, TARGET_TOTAL_EPISODES)
stdout_path = selected_run_dir / 'stdout.log'
stderr_path = selected_run_dir / 'stderr.log'
summary_path = selected_run_dir / 'run_summary.json'
status_path = selected_run_dir / 'run_status.json'
progress_path = selected_run_dir / 'progress.json'

print(f'run_dir={selected_run_dir}')
print('\n=== run_status ===')
print(json.dumps(read_json_if_exists(status_path), indent=2) if read_json_if_exists(status_path) else '<missing>')
print('\n=== progress ===')
print(json.dumps(read_json_if_exists(progress_path), indent=2) if read_json_if_exists(progress_path) else '<missing>')
print('\n=== summary ===')
print(read_text_if_exists(summary_path) or '<run_summary.json not written yet>')
print('\n=== stdout tail ===')
print(tail_text(stdout_path, lines=60))
print('\n=== stderr tail ===')
print(tail_text(stderr_path, lines=60))


In [ ]:
summaries = load_summaries()
status_rows = load_status_rows()
current_target_total = latest_target_total()
current_status = [row for row in status_rows if row['target_total_episodes'] == current_target_total]
current_completed = [row for row in summaries if row['target_total_episodes'] == current_target_total]

print(f'latest_target_total={current_target_total}')
print('\n=== current status rows ===')
as_table(current_status, sort_by=['seed'], ascending=True)

print('\n=== completed summaries at this target ===')
as_table(current_completed, sort_by=['seed'], ascending=True)


In [ ]:
SEED = 7
TARGET_TOTAL_EPISODES = latest_target_total(seed=SEED)
curve_path = latest_curve_path(SEED, TARGET_TOTAL_EPISODES)
curve_text = read_text_if_exists(curve_path)

if plt is None:
    print('matplotlib is not installed in this kernel; skipping curve plot.')
elif not curve_text:
    print(f'No curve written yet for {curve_path}')
else:
    curve_rows = list(csv.DictReader(io.StringIO(curve_text)))
    episodes = [int(float(row['episode'])) for row in curve_rows]
    reward_ma_10 = [float(row['reward_ma_10']) for row in curve_rows]
    reward_sum = [float(row['reward_sum']) for row in curve_rows]

    plt.figure(figsize=(10, 4))
    plt.plot(episodes, reward_ma_10, label='reward_ma_10', linewidth=2)
    plt.plot(episodes, reward_sum, label='reward_sum', alpha=0.35)
    plt.title(f'current run seed {SEED} target {TARGET_TOTAL_EPISODES}')
    plt.xlabel('reported training episode')
    plt.ylabel('reward')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


In [ ]:
LATEST_TARGET_TOTAL = latest_target_total()
print(f'volume={VOLUME_NAME}')
print(f'latest_target_total={LATEST_TARGET_TOTAL}')


## Initial 800-Run Comparison

These cells display the original seed 7 vs seed 11 analysis figures from the first local 800-episode auxhead-lite comparison. The PNG artifacts are now stored in-repo under `/Users/stephenbeale/Projects/ToM AI Research Team/modal/initial-800-run-analysis`.


In [ ]:
from IPython.display import Image, Markdown, display

INITIAL_800_FIGURES = [
    ('Belief transitions (counts)', 'belief-change-heatmap.png'),
    ('Belief transitions (normalized)', 'normalised transitions heatmap.png'),
    ('Normalized transition delta (seed11 - seed7)', 'normalised-transition-delta-seed-7-11.png'),
    ('Mean belief confidence by step index', 'mean-belief-confidence-by-seed.png'),
    ('Outcome rate by seed', 'outcome-rate-by-seed.png'),
    ('Action distribution by seed', 'action-distribution-by-seed.png'),
]

for title, filename in INITIAL_800_FIGURES:
    figure_path = INITIAL_800_ANALYSIS_ROOT / filename
    display(Markdown(f'### {title}'))
    display(Image(filename=str(figure_path), width=1100))


## Local 130k Comparison

These cells read the in-project local Modal outputs under `/Users/stephenbeale/Projects/ToM AI Research Team/modal/tom-130k-modal-results` and show the generated verdict/charts from `scripts/modal_130k_report.py`.


In [ ]:
from IPython.display import SVG, Markdown, display

LOCAL_REPORTS_ROOT = RESULTS_130_ROOT / 'reports'

display(Markdown((LOCAL_REPORTS_ROOT / 'modal_130k_verdict.md').read_text()))
summary_json = json.loads((LOCAL_REPORTS_ROOT / 'modal_130k_summary.json').read_text())
summary_json['verdicts']


In [ ]:
display(SVG(filename=str(LOCAL_REPORTS_ROOT / "metric_comparison.svg")))
display(SVG(filename=str(LOCAL_REPORTS_ROOT / "curve_overlay.svg")))
display(SVG(filename=str(LOCAL_REPORTS_ROOT / "family_outcome_grid.svg")))


## Local 140k Comparison

These cells read the completed in-project 140k report pack under `/Users/stephenbeale/Projects/ToM AI Research Team/modal/tom-140k-modal-results/reports`.


In [ ]:
from IPython.display import SVG, Markdown, display

LOCAL_140_REPORTS_ROOT = RESULTS_140_OLD_ROOT / 'reports'

display(Markdown((LOCAL_140_REPORTS_ROOT / 'modal_longrun_verdict.md').read_text()))
longrun_summary_json = json.loads((LOCAL_140_REPORTS_ROOT / 'modal_longrun_summary.json').read_text())
longrun_summary_json['verdicts']


In [ ]:
display(SVG(filename=str(LOCAL_140_REPORTS_ROOT / "longrun_metric_lines.svg")))
display(SVG(filename=str(LOCAL_140_REPORTS_ROOT / "longrun_curve_overlay.svg")))
display(SVG(filename=str(LOCAL_140_REPORTS_ROOT / "longrun_family_success.svg")))
